In [3]:
import numpy as np
import torch
import matplotlib.pyplot as plt

from principal_agent_mdp import PrincipalAgentMDP
from principal_dq import PrincipalDQ
from agent_dq import AgentDQ

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [4]:
def rollout(env, principal, agent, q_theta, Q_phi_bar, rho, episodes=50, T=200, gamma=1.0):
    def eval_episode():
        s = env.s0
        G_p, G_a = 0.0, 0.0
        discount = 1.0

        for t in range(T):
            a_p = principal.induce_action(s, q_theta)
            o = env.sample_outcome(s, a_p)
            s_next = env.T(s, o)

            b = principal.find_best_contract(s, a_p, Q_phi_bar(...))
            r_p = env.R_principal(s, b, o)
            r_a = env.R_agent(s, a_p, b, o)

            G_p += discount * r_p
            G_a += discount * r_a

            discount *= gamma
            s = s_next

            if env.is_terminal(s):
                break

        return G_p, G_a

    P, A = [], []
    for _ in range(episodes):
        p, a = eval_episode()
        P.append(p)
        A.append(a)

    return np.mean(P), np.std(P), np.mean(A), np.std(A)


mdp = PrincipalAgentMDP(gamma=1.0)

principal_dqn = PrincipalDQ(mdp, mdp.r_p, epsilon=0.0)
agent_dqn = AgentDQ(mdp, epsilon=0.0)

U_p_dqn, std_p_dqn, U_a_dqn, std_a_dqn = rollout(
    mdp,
    principal_dqn,
    agent_dqn,
    q_theta,
    Q_phi_bar,
    rho_dqn,
)

principal_star = PrincipalDQ(mdp, mdp.r_p, epsilon=0.0)
agent_star = AgentDQ(mdp, epsilon=0.0)

U_p_star, std_p_star, U_a_star, std_a_star = rollout(
    mdp,
    principal_star,
    agent_star,
    q_theta=None,   # or oracle policy wrapper if needed
    Q_phi_bar=None,
    rho=rho_star
)


regret_p = U_p_star - U_p_dqn
regret_a = U_a_star - U_a_dqn


labels = ["Principal", "Agent"]

dqn_vals = [U_p_dqn, U_a_dqn]
star_vals = [U_p_star, U_a_star]

x = np.arange(len(labels))

plt.bar(x - 0.2, dqn_vals, 0.4, label="DQN")
plt.bar(x + 0.2, star_vals, 0.4, label="Oracle")

plt.xticks(x, labels)
plt.legend()
plt.title("Utility Comparison")
plt.show()


print("Principal regret:", regret_p)
print("Agent regret:", regret_a)
print("Welfare regret:", regret_w)

NameError: name 'q_theta' is not defined